# Adatok felderítése, pipeline kialakítása

1: SZÜKSÉGES CSOMAGOK IMPORTÁLÁSA

In [5]:
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import datetime

kelvin = 273.15

2: DATA EXPLORATORY

In [6]:
files = sorted(Path(r"tigge_data").glob("*.grib"))
print(f"- Összesen {len(files)} fájlt tudtunk letölteni")

# kiválasztunk egy mintát
ds = xr.open_dataset(files[0], engine="cfgrib")

print()
print("- Nézzünk bele egy grib fájlba:")
print(ds)

- Összesen 158 fájlt tudtunk letölteni

- Nézzünk bele egy grib fájlba:
<xarray.Dataset> Size: 195kB
Dimensions:            (time: 31, step: 4, values: 97)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    latitude           (values) float64 776B ...
    longitude          (values) float64 776B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    surface            float64 8B ...
Dimensions without coordinates: values
Data variables:
    mx2t6              (time, step, values) float32 48kB ...
    mn2t6              (time, step, values) float32 48kB ...
    t2m                (time, step, values) float32 48kB ...
    tp                 (time, step, values) float32 48kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  E

In [7]:
cities = {
    "Budapest": (47.4979, 19.0402),
    "Debrecen": (47.5316, 21.6273),
    "Pécs":     (46.0727, 18.2323),
    "Szeged":   (46.2530, 20.1414),
    "Szombathely": (47.23,16.62)
}

In [8]:
print(ds.latitude.values)

[48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.763994   48.763994   48.763994   48.763994   48.763994   48.763994
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 48.3145566  48.3145566  48.3145566  48.3145566  48.3145566  48.3145566
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918 47.86511918
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175 47.41568175
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431 46.96624431
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685 46.51680685
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938 46.06736938
 46.06736938 46.06736938 46.06736938 46.06736938 46.06736

In [9]:
def nearest_point(ds, cel_latitude, cel_longitude):
    lat = ds.latitude.values
    lon = ds.longitude.values
    distance = (lat - cel_latitude)**2 + (lon - cel_longitude)**2
    idx = np.argmin(distance)
    return ds.isel(values=idx)

In [10]:
print(nearest_point(ds,cities["Budapest"][0],cities["Budapest"][1]))

<xarray.Dataset> Size: 3kB
Dimensions:            (time: 31, step: 4)
Coordinates:
  * time               (time) datetime64[ns] 248B 2006-10-01 ... 2006-10-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 992B ...
    number             int64 8B ...
    heightAboveGround  float64 8B ...
    latitude           float64 8B 47.42
    longitude          float64 8B 18.75
    surface            float64 8B ...
Data variables:
    mx2t6              (time, step) float32 496B ...
    mn2t6              (time, step) float32 496B ...
    t2m                (time, step) float32 496B ...
    tp                 (time, step) float32 496B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Ra

In [11]:
print(ds.step.values)

[21600000000000 43200000000000 64800000000000 86400000000000]


3: EGY HÓNAP DATASET-JÉNEK ELŐÁLLÍTÁSA

In [12]:
def daily_aggregation(pont_ds):
    """pont_ds: egy adott város rácspontjára kivágott Dataset (time, step dimenziókkal)"""
    
    daily_max  = pont_ds["mx2t6"].max(dim="step")
    daily_min  = pont_ds["mn2t6"].min(dim="step")
    daily_mean_maxmin = (daily_max + daily_min) / 2
    daily_mean_t2m    = pont_ds["t2m"].mean(dim="step")
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    daily_precipitation  = pont_ds["tp"].isel(step=-1)

    
    df = pd.DataFrame({
        "datum": pont_ds["time"].values,
        "tmax": daily_max.values,
        "tmin": daily_min.values,
        "tkozep_maxmin": daily_mean_maxmin.values,
        "tkozep_t2m": daily_mean_t2m.values,
        "csapadek_mm": daily_precipitation.values,
    })
    return df

In [13]:
budapest_point = nearest_point(ds, cities["Budapest"][0], cities["Budapest"][1])
df = daily_aggregation(budapest_point)

In [14]:
print(df.head(10))

       datum        tmax        tmin  tkozep_maxmin  tkozep_t2m   csapadek_mm
0 2006-10-01  296.509338  284.494263     290.501801  290.405701 -9.024809e-09
1 2006-10-02  296.138947  285.307709     290.723328  290.889587  4.577637e-02
2 2006-10-03  297.855621  289.658508     293.757080  294.179932 -3.931024e-09
3 2006-10-04  294.084991  286.512787     290.298889  288.218964  3.067017e+00
4 2006-10-05  291.883026  284.959686     288.421356  287.318237 -1.902598e-09
5 2006-10-06  290.791290  280.276398     285.533844  284.847534 -1.338483e-11
6 2006-10-07  291.457367  279.635254     285.546326  285.808685 -4.280513e-16
7 2006-10-08  291.422852  282.029755     286.726318  286.262329  5.340576e-02
8 2006-10-09  291.429108  278.670624     285.049866  285.901550 -2.466796e-10
9 2006-10-10  292.332672  278.916870     285.624756  285.218933 -1.340083e-09


Látjuk hogy kicsit furák az adatok. A hőmérséklet Kelvinben van megadva, a csapadéknál pedig apró, numerikus zajokat figyelhetünk meg.
Próbaljuk ezt megoldani!

In [15]:
def daily_aggregation(pont_ds):

    daily_max  = pont_ds["mx2t6"].max(dim="step") - kelvin
    daily_min  = pont_ds["mn2t6"].min(dim="step") - kelvin
    daily_mean_maxmin = (daily_max + daily_min) / 2
    daily_mean_t2m    = pont_ds["t2m"].mean(dim="step") - kelvin
    
    # csapadék: kumulatív, a 24h (utolsó) lépés adja a teljes napi összeget
    daily_precipitation = (pont_ds["tp"].isel(step=-1)).clip(min = 0)
    
    df = pd.DataFrame({
            "datum": pont_ds["time"].values,
            "d_ta_maxmin": daily_mean_maxmin.values,
            "d_ta_t2m": daily_mean_t2m.values,
            "d_tx": daily_max.values,
            "d_tn": daily_min.values,
            "csapadek": daily_precipitation.values,
        })
    return df

In [16]:
df2 = daily_aggregation(budapest_point)
print(df2)

        datum  d_ta_maxmin   d_ta_t2m       d_tx       d_tn  csapadek
0  2006-10-01    17.351807  17.255707  23.359344  11.344269  0.000000
1  2006-10-02    17.573334  17.739594  22.988953  12.157715  0.045776
2  2006-10-03    20.607071  21.029938  24.705627  16.508514  0.000000
3  2006-10-04    17.148895  15.068970  20.934998  13.362793  3.067017
4  2006-10-05    15.271362  14.168243  18.733032  11.809692  0.000000
5  2006-10-06    12.383850  11.697540  17.641296   7.126404  0.000000
6  2006-10-07    12.396317  12.658691  18.307373   6.485260  0.000000
7  2006-10-08    13.576309  13.112335  18.272858   8.879761  0.053406
8  2006-10-09    11.899872  12.751556  18.279114   5.520630  0.000000
9  2006-10-10    12.474777  12.068939  19.182678   5.766876  0.000000
10 2006-10-11    11.932541  11.035919  18.664978   5.200104  0.000000
11 2006-10-12    12.838684  11.597595  18.474213   7.203156  0.000000
12 2006-10-13    13.527634  14.220306  18.736877   8.318390  0.000000
13 2006-10-14    13.

Na, máris jófelé tartunk. Viszont találtam a met.hu-n egy leírást ami kicsit árnyalja a helyzetünket.
[met.hu](https://met.hu/eghajlat/magyarorszag_eghajlata/eghajlati_adatsorok/Budapest/leirasok/szamitasok/)

4: MET SZERINTI MÉRÉSI SZABVÁNYOK IMPLEMENTÁLÁSA

A met.hu napi számítási procedúrái kicsit bonyolultabbak. A napi felbontásnál az adott nap értékeinek kiszámításáHoz szükségünk van az azt megelőző és az utána levő nap adataira is. Ez értelemszerűen gondot okoz a hónapok utolsó és első napjainak számításánál mert az adatok hónapos felbontásban vannak. Ezért a grib fájlokat úgy kell megnyitni, hogy két szomszédos hónapot összefésülünk.

A dátum-intervallumunk legelső és legutolsó napját viszont nem tudjuk így lekezelni, ezt el kell fogadnunk.

In [17]:
def get_month_context_range(year, month, debug = False):
    prev_year, prev_month = (year - 1, 12) if month == 1 else (year, month - 1)
    prev_file = Path(f"tigge_data/{prev_year}-{prev_month:02}_cf.grib")

    next_year, next_month = (year + 1, 1) if month == 12 else (year, month + 1)
    next_file = Path(f"tigge_data/{next_year}-{next_month:02}_cf.grib")

    fname = Path(f"tigge_data/{year}-{month:02}_cf.grib")
    
    if not fname.exists():
        if debug : print(f"[KIHAGYVA] A hónap hiányzik: {fname}")
        return None
    
    file = xr.open_dataset(fname, engine="cfgrib")
    parts = [file]

    if prev_file.exists():
        prev = xr.open_dataset(prev_file, engine="cfgrib").isel(time=-1)
        parts.insert(0, prev.expand_dims("time"))
    else:
        if debug: print(f"[FIGYELEM] Nincs meg az előző hónap fájlja: {prev_file.name}")

    if next_file.exists():
        nxt = xr.open_dataset(next_file, engine="cfgrib").isel(time=0)
        parts.append(nxt.expand_dims("time"))
    else:
        if debug: print(f"[FIGYELEM] Nincs meg a következő hónap fájlja: {next_file.name}")

    merged = xr.concat(parts, dim="time")
    return merged.sortby("time")

In [20]:
teszt = get_month_context_range(2006,10)

In [21]:
print(teszt.sizes["time"])

print()
print("Na király.")

31

Na király.


In [22]:
# Napi max és min hőmérséklet

def daily_max_min_methu(pont_ds, day_index):

    prev_day_24h_max = pont_ds["mx2t6"].isel(time = day_index - 1, step = 3)
    current_day_6_12_24_max = pont_ds["mx2t6"].isel(time = day_index, step=[0, 1, 2])
    current_day_max = current_day_6_12_24_max.max()

    daily_max = max(float(prev_day_24h_max), float(current_day_max)) - kelvin


    prev_day_24h_min = pont_ds["mn2t6"].isel(time = day_index - 1, step = 3)
    current_day_6_12_24_min = pont_ds["mn2t6"].isel(time=day_index, step = [0, 1, 2])
    current_day_min = current_day_6_12_24_min.min()

    daily_min = min(float(prev_day_24h_min), float(current_day_min)) - kelvin

    return [daily_max, daily_min]

In [23]:
print(daily_max_min_methu(budapest_point,1))

[22.988946533203148, 12.157708740234398]


In [24]:
# Napi csapadék
def daily_precipitation_methu(pont_ds, day_index):

    current_step6 = float(pont_ds["tp"].isel(time = day_index, step = 0))
    current_step24 = float(pont_ds["tp"].isel(time = day_index, step = 3))
    next_step6 = float(pont_ds["tp"].isel(time = day_index + 1, step = 0))

    precipitation_m = (current_step24 - current_step6) + next_step6

    return max(precipitation_m, 0)


In [25]:
print(daily_precipitation_methu(budapest_point, 8))

1.205731377617897e-09


In [26]:
def daily_aggregation_methu(pont_ds, year, month):
    n_days = pont_ds.sizes["time"]
    
    rows = []
    for day_index in range(n_days):
        datum = pont_ds["time"].isel(time=day_index).values
        
        if day_index - 1 < 0 or day_index + 1 >= n_days:
            nap = pont_ds.isel(time=day_index)
            d_tx = float(nap["mx2t6"].max(dim="step")) - kelvin
            d_tn = float(nap["mn2t6"].min(dim="step")) - kelvin
            csapadek = max(float(nap["tp"].isel(step=-1)), 0)
            
        else:
            d_tx, d_tn = daily_max_min_methu(pont_ds, day_index)
            csapadek = daily_precipitation_methu(pont_ds, day_index)
        
        d_ta = (d_tx + d_tn) / 2 if not (np.isnan(d_tx) or np.isnan(d_tn)) else np.nan
        
        rows.append({
            "datum": pd.Timestamp(datum),
            "d_ta": round(d_ta,1),
            "d_tx": round(d_tx,1),
            "d_tn": round(d_tn,1),
            "d_rs": round(csapadek,1),
        })
    
    df = pd.DataFrame(rows)
    
    # csak az adott hónap saját napjai maradjanak - a becsatolt szomszéd-napokat dobjuk el
    df = df[(df["datum"].dt.year == year) & (df["datum"].dt.month == month)].reset_index(drop=True)

    df["datum"] = df["datum"].astype(str).str[0:10]
    
    return df

In [27]:
budapest_point = nearest_point(teszt, cities["Budapest"][0], cities["Budapest"][1])

df3 = daily_aggregation_methu(budapest_point, 2008, 10)
print(df3)

Empty DataFrame
Columns: [datum, d_ta, d_tx, d_tn, d_rs]
Index: []


Na akkor ott tartunk, hogy egy város adott év adott hónapjának le tudjuk szedni az adatait, amit akarunk. Nézzük meg, mindegyik várost akkor.

In [28]:
for city in cities:

    lat = cities[city][0]
    lon = cities[city][1]

    pt = nearest_point(teszt, lat, lon)

    df = daily_aggregation_methu(pt, 2008, 10)

    print(f"City: {city}")
    print(df.head(3))
    print('-------')

City: Budapest
Empty DataFrame
Columns: [datum, d_ta, d_tx, d_tn, d_rs]
Index: []
-------
City: Debrecen
Empty DataFrame
Columns: [datum, d_ta, d_tx, d_tn, d_rs]
Index: []
-------
City: Pécs
Empty DataFrame
Columns: [datum, d_ta, d_tx, d_tn, d_rs]
Index: []
-------
City: Szeged
Empty DataFrame
Columns: [datum, d_ta, d_tx, d_tn, d_rs]
Index: []
-------
City: Szombathely
Empty DataFrame
Columns: [datum, d_ta, d_tx, d_tn, d_rs]
Index: []
-------


Na tök jó fele haladunk. Innen tovább:

-> Ne csak egy hónapra szóljanak a df-ek

-> CSV fájlá alakítás tanulmányozása

-> Egész adatkinyerés automatizációja

In [29]:
teszt2 = get_month_context_range(2014,9)

In [30]:
for city in cities:

    lat = cities[city][0]
    lon = cities[city][1]

    pt = nearest_point(teszt2, lat, lon)

    df = daily_aggregation_methu(pt, 2014, 9)

    print(f"City: {city}")
    print(df.head(3))
    print('-------')

City: Budapest
        datum  d_ta  d_tx  d_tn  d_rs
0  2014-09-01  19.9  23.7  16.0   3.9
1  2014-09-02  20.1  24.7  15.5   0.0
2  2014-09-03  22.1  25.6  18.6   0.0
-------
City: Debrecen
        datum  d_ta  d_tx  d_tn  d_rs
0  2014-09-01  21.1  26.9  15.4   0.3
1  2014-09-02  22.1  27.4  16.7   0.1
2  2014-09-03  23.1  27.1  19.1   0.0
-------
City: Pécs
        datum  d_ta  d_tx  d_tn  d_rs
0  2014-09-01  20.9  25.6  16.3  15.6
1  2014-09-02  17.6  20.4  14.7   1.9
2  2014-09-03  19.8  23.3  16.3   7.7
-------
City: Szeged
        datum  d_ta  d_tx  d_tn  d_rs
0  2014-09-01  22.5  26.4  18.7   1.9
1  2014-09-02  21.4  25.9  16.9   0.1
2  2014-09-03  23.0  27.6  18.5   1.5
-------
City: Szombathely
        datum  d_ta  d_tx  d_tn  d_rs
0  2014-09-01  15.7  19.0  12.4  34.4
1  2014-09-02  14.9  15.6  14.2   9.2
2  2014-09-03  17.8  21.0  14.7   7.2
-------


5: TXT-be írás PRÓBÁLKOZÁS

A met.hu-ról nem csv fájlba, hanem txt formátumba lehet letölteni az anyagot ezért énis ezt fogom csinálni

In [31]:
TXT_DIRECTORY = Path("out_txt") # output mappa létrehozása
TXT_DIRECTORY.mkdir(exist_ok=True)

In [32]:
print(df.head(1))  # a df amin tudunk kisérletezni

        datum  d_ta  d_tx  d_tn  d_rs
0  2014-09-01  15.7  19.0  12.4  34.4


In [33]:
for idx, row in df.iterrows():
    print(row["datum"], row["d_ta"], row["d_tx"], row["d_tn"], row["d_rs"])

2014-09-01 15.7 19.0 12.4 34.4
2014-09-02 14.9 15.6 14.2 9.2
2014-09-03 17.8 21.0 14.7 7.2
2014-09-04 20.0 24.8 15.2 1.4
2014-09-05 19.8 23.7 16.0 1.7
2014-09-06 18.7 21.4 15.9 1.7
2014-09-07 18.1 22.7 13.5 0.0
2014-09-08 18.6 23.7 13.5 0.2
2014-09-09 20.6 26.0 15.2 10.5
2014-09-10 18.6 20.8 16.5 1.3
2014-09-11 16.1 17.8 14.4 35.8
2014-09-12 15.0 17.0 12.9 21.6
2014-09-13 16.4 20.6 12.3 9.0
2014-09-14 16.0 17.6 14.5 7.3
2014-09-15 16.8 19.5 14.2 7.6
2014-09-16 15.8 20.4 11.2 0.0
2014-09-17 15.9 21.2 10.6 0.0
2014-09-18 16.3 20.8 11.9 0.0
2014-09-19 18.0 22.9 13.1 0.0
2014-09-20 18.3 24.3 12.3 1.3
2014-09-21 19.8 24.5 15.1 22.3
2014-09-22 15.8 17.7 14.0 0.1
2014-09-23 12.2 15.2 9.2 0.4
2014-09-24 11.3 16.9 5.6 0.0
2014-09-25 12.2 15.9 8.5 0.5
2014-09-26 13.0 15.1 10.9 4.4
2014-09-27 15.7 18.7 12.8 0.7
2014-09-28 14.3 19.5 9.2 0.0
2014-09-29 15.4 20.3 10.5 0.0
2014-09-30 14.3 19.9 8.8 0.0


In [34]:
with open("out_txt/proba.txt", "w") as f:

    f.write("#datum;d_ta;d_tx;d_tn;d_rs\n")

    for idx, row in df.iterrows():
        f.write(f"{row["datum"]};{row["d_ta"]};{row["d_tx"]};{row["d_tn"]};{row["d_rs"]}\n")

Ezzel már egy hónap adatait a helyes formátumban ki tudjuk íratni egy városra

Probáljunk meg több hónapot összerakni

In [35]:
def month_breakdown(start, end):

    year, month = start
    while (year, month) <= end:
        yield year, month

        month += 1
        if month > 12:
            month = 1
            year += 1

In [36]:
def output_multiple_months(start,end, city, debug = False):
        
        months = month_breakdown(start,end)
        skipped = []

        with open("out_txt/bp_output2.txt", "w") as f:

                f.write("#datum;d_ta;d_tx;d_tn;d_rs\n")

                for month in months:
                        context = get_month_context_range(month[0],month[1], debug)

                        if context is None:
                                skipped.append(month)
                                continue

                        pt = nearest_point(context,cities[city][0],cities[city][1])

                        df = daily_aggregation_methu(pt, month[0],month[1])

                        for idx, row in df.iterrows():
                                f.write(f"{row["datum"]};{row["d_ta"]};{row["d_tx"]};{row["d_tn"]};{row["d_rs"]}\n")

        f.close()

In [37]:
output_multiple_months((2010,1), (2010,3), "Budapest", True)

DatasetBuildError: multiple values for unique key, try re-open the file with one of:
    filter_by_keys={'numberOfPoints': 97}
    filter_by_keys={'numberOfPoints': 232}